# Brief and clinical conditions — DP and DA (Python)

Last run: 2026-07-27.

Ports `R DP and DA - Brief and clinical conditions.Rmd`, fixing bugs found in it: a destructive df overwrite, an NA-filter applied after sampling instead of before, mixed age columns, triplicated per-instrument chunks, and post hocs that ignored the age covariate. Added §4 (item specificity), which was absent from the Rmd.

Brief-Spec A (`tead_*`) analytic sample: `aq_total <= 8` plus a gender-stratified draw (up to 350 male), needed to reach N=700 with a balanced gender split. Brief-Spec F (`team_*`): no AQ filter and no gender stratification — the sample has no male respondents, and the filter only suppressed ASD representation without serving any other purpose.

Draws are seeded (`random_state=123`) and reproducible here, but differ from the rows R's `set.seed(123)` selected.

## 1. Setup

Analysis stack: `pandas` for data handling, `rdata` to read the published `.RData`, `statsmodels`/`scipy` for models and tests.

In [1]:
import tempfile, urllib.request, warnings

import numpy as np
import pandas as pd
import rdata
import statsmodels.formula.api as smf
from rdata.parser import RObjectType
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.contingency_tables import Table2x2, mcnemar

pd.set_option("display.max_columns", None)

## 2. Data preparation (DP)

### 2.1 Get data

Downloads the open data from the previous publication (OSF). The `.RData` holds a whole R workspace (including lavaan S4 objects) that generic converters choke on, so the loader walks the workspace pairlist iteratively and converts **only** the two study data frames.

In [2]:
def load_rdata_frames(url, names):
    # walk the workspace pairlist and convert only the requested data frames
    node = rdata.parser.parse_file(
        urllib.request.urlretrieve(url, tempfile.gettempdir() + "/brief_clinical.RData")[0]
    ).object
    out = {}
    while getattr(node, "info", None) is not None and node.info.type == RObjectType.LIST:
        car, cdr = node.value
        # object names are symbol tags; repeated symbols are stored as references
        tag = node.tag
        if tag is not None and tag.info.type == RObjectType.REF:
            tag = tag.referenced_object
        name = tag.value.value if tag is not None and tag.value is not None else None
        name = name.decode() if isinstance(name, bytes) else name
        if name in names:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                out[name] = rdata.conversion.convert(car)
        node = cdr
    return [out[n] for n in names]

df_tead, df_team = load_rdata_frames("https://osf.io/download/87kqz", ["df_tead", "df_team"])
df_tead.shape, df_team.shape

((3302, 129), (7850, 88))

### 2.2 Items and subscales

The subscale → items maps are the single source of truth; the flat item lists are derived from them (the Rmd kept items and totals in two separate places that had to agree by hand).

Brief-Spec A item labels used in the specificity tables:

- 8: I have trouble reading facial expressions
- 35: I find it hard to read other people's body language
- 36: I often miss social cues.
- 38: I find it hard to know when someone wants me to do something unless they tell me directly
- 33: I have difficulty understanding non-literal language, such as irony, sarcasm, idioms, metaphors, or figurative speech
- 9: Compared to other people, I often feel out of place in social interactions
- 29: Others have told me I seem withdrawn or distant in social interactions.
- 3: I often become exhausted from social interactions.
- 28: I enjoy social interaction less than other people do.
- 5: I often try to avoid interacting with people in social situations.
- 70: I become intensely focused on certain topics, sometimes without a clear reason.
- 69: When I am engaged in my favorite interest, I often become so absorbed that I ignore everything else.
- 66: I often become so deeply engrossed in one activity that I become unaware of my surroundings.
- 4: I try to hide behaviors that might be viewed as unusual by other people (e.g., rocking, hand-wringing, pacing).
- 73: I am more sensitive to sounds, lights, and touches than other people are.
- 52: I find it difficult to deal with changes in my daily routine.
- 60: Unexpected changes to my plans make me feel uncomfortable.
- 59: I find it difficult to adjust to unexpected changes.
- 56: I have certain routines that I must follow.
- 63: Others have told me that I over-react to minor changes in my environment

Brief-Spec F item labels used in the specificity tables:

Camouflage: Compensation and assimilation
- 6: I imitate other people when I am interacting with them.
- 11: I imitate other people's hand gestures when I am interacting with them.
- 1: I imitate other people's behavior when I am interacting with them.
- 9: I imitate how people talk or speak when I am interacting with them.

Gender self-identification
- 3: I found it very difficult to make friends as a child (ages 5-12).
- 4: As a child (ages 5-12), I was not interested in playing with other children.
- 5: As a child (ages 5-12), other children thought my interests were unusual.
- 25: As a child (ages 5-12), I felt unsure about how to behave in social situations.

Camouflaging: Masking
- 22: I often "pretend" to be like other people.
- 14: I often feel that I am "playing a role" in social situations.
- 21: I often hide my "true self" in social situations.
- 7: I have to behave in a specific way for others to accept me.

Sensory sensitivity
- 37: Personal grooming activities (e.g., combing hair, brushing teeth, washing myself) make me feel uncomfortable.
- 36: Everyday activities (e.g., shopping, grocery store, bank) make me feel overwhelmed.
- 32: Doing new things makes me feel anxious.
- 35: I am more sensitive to sounds, lights, and touches than other people are.

In [3]:
subscales_tead = {
    "social_communication": ["tead_8", "tead_35", "tead_36", "tead_38", "tead_33"],
    "social_interaction":   ["tead_9", "tead_3", "tead_5", "tead_28", "tead_29"],
    "sensorial":            ["tead_70", "tead_69", "tead_4", "tead_73", "tead_66"],
    "repetitive_behavior":  ["tead_52", "tead_59", "tead_60", "tead_63", "tead_56"],
}
subscales_team = {
    "camouflage_ca":       ["team_6", "team_11", "team_1", "team_9"],
    "gender_self_id":      ["team_5", "team_3", "team_4", "team_25"],
    "camouflage_m":        ["team_14", "team_22", "team_7", "team_21"],
    "sensory_sensitivity": ["team_36", "team_37", "team_32", "team_35"],
}

# flat item lists, derived once from the maps above
items_tead = [i for v in subscales_tead.values() for i in v]
items_team = [i for v in subscales_team.values() for i in v]

In [4]:
# TEAD and TEAM item labels used in the specificity tables
item_labels_tead = {
    "tead_8": "8 - I have trouble reading facial expressions",
    "tead_35": "35 - I find it hard to read other people's body language",
    "tead_36": "36 - I often miss social cues.",
    "tead_38": "38 - I find it hard to know when someone wants me to do something unless they tell me directly",
    "tead_33": "33 - I have difficulty understanding non-literal language, such as irony, sarcasm, idioms, metaphors, or figurative speech",
    "tead_9": "9 - Compared to other people, I often feel out of place in social interactions",
    "tead_29": "29 - Others have told me I seem withdrawn or distant in social interactions.",
    "tead_3": "3 - I often become exhausted from social interactions.",
    "tead_28": "28 - I enjoy social interaction less than other people do.",
    "tead_5": "5 - I often try to avoid interacting with people in social situations.",
    "tead_70": "70 - I become intensely focused on certain topics, sometimes without a clear reason.",
    "tead_69": "69 - When I am engaged in my favorite interest, I often become so absorbed that I ignore everything else.",
    "tead_66": "66 - I often become so deeply engrossed in one activity that I become unaware of my surroundings.",
    "tead_4": "4 - I try to hide behaviors that might be viewed as unusual by other people (e.g., rocking, hand-wringing, pacing).",
    "tead_73": "73 - I am more sensitive to sounds, lights, and touches than other people are.",
    "tead_52": "52 - I find it difficult to deal with changes in my daily routine.",
    "tead_60": "60 - Unexpected changes to my plans make me feel uncomfortable.",
    "tead_59": "59 - I find it difficult to adjust to unexpected changes.",
    "tead_56": "56 - I have certain routines that I must follow.",
    "tead_63": "63 - Others have told me that I over-react to minor changes in my environment",
}

item_labels_team = {
    "team_6": "6 - I imitate other people when I am interacting with them.",
    "team_11": "11 - I imitate other people's hand gestures when I am interacting with them.",
    "team_1": "1 - I imitate other people's behavior when I am interacting with them.",
    "team_9": "9 - I imitate how people talk or speak when I am interacting with them.",
    "team_3": "3 - I found it very difficult to make friends as a child (ages 5-12).",
    "team_4": "4 - As a child (ages 5-12), I was not interested in playing with other children.",
    "team_5": "5 - As a child (ages 5-12), other children thought my interests were unusual.",
    "team_25": "25 - As a child (ages 5-12), I felt unsure about how to behave in social situations.",
    "team_22": "22 - I often \"pretend\" to be like other people.",
    "team_14": "14 - I often feel that I am \"playing a role\" in social situations.",
    "team_21": "21 - I often hide my \"true self\" in social situations.",
    "team_7": "7 - I have to behave in a specific way for others to accept me.",
    "team_37": "37 - Personal grooming activities (e.g., combing hair, brushing teeth, washing myself) make me feel uncomfortable.",
    "team_36": "36 - Everyday activities (e.g., shopping, grocery store, bank) make me feel overwhelmed.",
    "team_32": "32 - Doing new things makes me feel anxious.",
    "team_35": "35 - I am more sensitive to sounds, lights, and touches than other people are.",
}

### 2.3 Clinical groups

Self-reported diagnoses parsed by regex (matches the idiosyncratic free-text answers in this dataset). `classify` turns the three flags into mutually exclusive groups; Brief-Spec F additionally excludes two ambiguous free-text answers. Groups are coded on the full data, before sampling.

In [5]:
def has(col, pattern):
    # case-insensitive regex on free text; missing answers count as "no"
    return col.astype("string").str.contains(pattern, case=False, regex=True, na=False).astype(bool)

def classify(asd, adhd, ocd):
    # mutually exclusive clinical groups from the three diagnosis flags
    return np.select(
        [asd & ~adhd & ~ocd, asd & adhd & ~ocd, asd & ~adhd & ocd, asd & adhd & ocd,
         adhd & ~asd & ~ocd, adhd & ocd & ~asd, ocd & ~asd & ~adhd],
        ["asd", "asd_adhd", "asd_ocd", "all_conditions", "adhd", "adhd_ocd", "ocd"],
        default="control",
    )

df_tead = df_tead.assign(
    asd=lambda d: has(d.voce_tem_diagnostico_de_transtorno_do_espectro_autista_tea,
                      r"sim|concordo fortemente|diagnosticad[oa]|recebi|avalia[cç][aã]o neuropsicol[oó]gica.*autismo|1997.*asperger|psiquiatra.*espectro"),
    adhd=lambda d: has(d.voce_tem_diagnostico_de_transtorno_de_deficit_de_atencao_e_hiperatividade_tdah,
                       r"sim|concordo fortemente"),
    ocd=lambda d: has(d.ja_recebi_diagnostico_de_transtorno_obsessivo_compulsivo_toc,
                      r"sim|concordo|concordo fortemente|neurologista me perguntou se eu sabia que eu tenho|recebi antes de receber o diagnostico de autismo|recebi, mas nao concordo com ele"),
).assign(clinical_group=lambda d: classify(d.asd, d.adhd, d.ocd))

df_team = df_team.assign(
    asd=lambda d: has(d.voce_tem_diagnostico_de_transtorno_do_espectro_autista_tea,
                      r"sim|a medica|altas|estou.*autismo|nao confiei|so em 1997"),
    adhd=lambda d: has(d.ja_recebi_diagnostico_de_transtorno_de_deficit_de_atencao_e_hiperatividade_tdah,
                       r"sim|concordo|5|h[aá] 2 anos"),
    ocd=lambda d: has(d.ja_recebi_diagnostico_de_transtorno_obsessivo_compulsivo_toc,
                      r"sim|concordo|5|j[aá] recebi mas n[aã]o acredito"),
    # two answers the Rmd flags as unclassifiable -> excluded from any group
    exclude=lambda d: has(d.ja_recebi_diagnostico_de_transtorno_de_deficit_de_atencao_e_hiperatividade_tdah,
                          r'sim e nao\. na infancia fui "diagnosticada" mas atualmente nao sei')
                    | has(d.ja_recebi_diagnostico_de_transtorno_obsessivo_compulsivo_toc,
                          r"sim, mas a maioria das caracteristicas foi confundida pelos profissionais, eram esteriotipias"),
).assign(clinical_group=lambda d: np.where(d.exclude, None, classify(d.asd, d.adhd, d.ocd)))

df_tead.clinical_group.value_counts(dropna=False), df_team.clinical_group.value_counts(dropna=False)

(clinical_group
 control           2054
 asd                363
 adhd               279
 asd_adhd           258
 ocd                168
 asd_ocd             87
 all_conditions      50
 adhd_ocd            43
 Name: count, dtype: int64,
 clinical_group
 control           5232
 adhd               898
 asd                479
 ocd                453
 asd_adhd           415
 adhd_ocd           173
 asd_ocd            107
 all_conditions      91
 NaN                  2
 Name: count, dtype: int64)

### 2.4 Analytic samples (n = 700 each)

Eligibility: complete item responses and a valid clinical group, plus for Brief-Spec A an AQ-10 cutoff (`aq_total <= 8`), needed to reach 700 with a gender-balanced draw (up to 350 male). Brief-Spec F has no AQ filter and no gender stratification: the sample has no male respondents, and the filter only suppressed ASD representation. Analytic samples are new frames; the source data is never overwritten.

In [6]:
df_tead_s = (
    df_tead
    .loc[lambda d: d[items_tead].notna().all(axis=1) & (d.aq_total <= 8) & d.clinical_group.notna()]
    .assign(male=lambda d: d.genero == "Masculino")
    # stratified draw: up to 350 males, remainder from the other genders
    .pipe(lambda d: pd.concat([
        d.loc[d.male].sample(n=min(350, d.male.sum()), random_state=123),
        d.loc[~d.male].sample(n=700 - min(350, d.male.sum()), random_state=123),
    ]))
    .drop(columns="male")
)

df_team_s = (
    df_team
    .loc[lambda d: d[items_team].notna().all(axis=1) & d.clinical_group.notna()]
    .sample(n=700, random_state=123)
)

df_tead_s.clinical_group.value_counts(), df_team_s.clinical_group.value_counts()

(clinical_group
 control           479
 adhd               69
 asd                53
 ocd                40
 asd_adhd           37
 adhd_ocd            8
 asd_ocd             8
 all_conditions      6
 Name: count, dtype: int64,
 clinical_group
 control           467
 adhd               77
 ocd                41
 asd_adhd           38
 asd                37
 asd_ocd            16
 adhd_ocd           14
 all_conditions     10
 Name: count, dtype: int64)

### 2.5 Age and subscale totals

Numeric age parsed once per instrument (Brief-Spec A: `idade`; Brief-Spec F: `qual_e_a_sua_idade_em_numeros`). Subscale totals computed from the maps in 2.2, for both instruments.

In [7]:
def enrich(d, subscales, age_col):
    # numeric age + one total per subscale
    return d.assign(
        age=lambda x: pd.to_numeric(x[age_col].astype("string").str.extract(r"(\d+)")[0], errors="coerce"),
        **{k: (lambda x, v=v: x[v].sum(axis=1)) for k, v in subscales.items()},
    )

df_tead_s = enrich(df_tead_s, subscales_tead, "idade")
df_team_s = enrich(df_team_s, subscales_team, "qual_e_a_sua_idade_em_numeros")

df_tead_s[["age", *subscales_tead]].describe().round(2)

,age,social_communication,social_interaction,sensorial,repetitive_behavior
count,700.0,700.00,700.00,700.00,700.00
mean,37.31,15.61,19.83,18.79,18.91
std,11.64,5.11,4.73,4.81,4.82
min,18.0,5.00,5.00,5.00,5.00
25%,28.75,12.00,18.00,16.00,16.00
50%,36.0,16.00,21.00,20.00,20.00
75%,44.0,20.00,23.00,22.00,23.00
max,77.0,25.00,25.00,25.00,25.00


### 2.6 Education

Tabulation of educational status in the analytic sample.

In [8]:
education_map = {
    "ensino fundamental incompleto": "fundamental",
    "ensino fundamental completo": "fundamental",
    "ensino medio incompleto": "medio",
    "ensino medio completo": "medio",
    "ensino tecnico ou profissionalizante": "medio",
    "ensino superior incompleto": "superior",
    "ensino superior completo": "superior",
    "pos-graduacao (especializacao, mestrado, doutorado)": "pos-graduacao",
}

education_order = ["fundamental", "medio", "superior", "pos-graduacao"]

def add_level_of_education(d):
    return d.assign(
        level_of_education=lambda x: pd.Categorical(
            x.nivel_de_escolaridade_que_melhor_descreve_seu_status_educacional_atual.map(education_map),
            categories=education_order,
            ordered=True,
        )
    )

df_tead_s = add_level_of_education(df_tead_s)
df_team_s = add_level_of_education(df_team_s)

pd.concat(
    {
        "Brief-Spec-A": df_tead_s.level_of_education.value_counts(dropna=False),
        "Brief-Spec-F": df_team_s.level_of_education.value_counts(dropna=False),
    },
    axis=1,
)

,Brief-Spec-A,Brief-Spec-F
level_of_education,,
superior,313,296
pos-graduacao,256,315
medio,108,81
fundamental,23,8


In [9]:
df_tead_s

,carimbo_de_data_hora,endereco_de_e_mail,nome_completo,pontuacao,x,qual_e_a_sua_idade_em_numeros,com_qual_genero_voce_se_identifica,estado_civil,nivel_de_escolaridade_que_melhor_descreve_seu_status_educacional_atual,voce_tem_diagnostico_de_transtorno_do_espectro_autista_tea,ja_recebi_diagnostico_de_transtorno_obsessivo_compulsivo_toc,tead_1,tead_2,tead_3,tead_4,tead_5,tead_6,tead_7,tead_8,tead_9,tead_10,tead_11,tead_12,tead_13,tead_14,tead_15,tead_16,tead_17,tead_18,tead_19,tead_20,tead_21,tead_22,tead_23,tead_24,tead_25,tead_26,tead_27,tead_28,tead_29,tead_30,tead_31,tead_32,tead_33,tead_34,tead_35,tead_36,tead_37,tead_38,tead_39,tead_40,tead_41,tead_42,tead_43,tead_44,tead_45,tead_46,tead_47,tead_48,tead_49,tead_50,tead_51,tead_52,tead_53,tead_54,tead_55,tead_56,tead_57,tead_58,tead_59,tead_60,tead_61,tead_62,tead_63,tead_64,tead_65,tead_66,tead_67,tead_68,tead_69,tead_70,tead_71,tead_72,tead_73,tead_74,tead_75,tead_76,tead_77,tead_78,tead_79,tead_80,gostaria_de_comentar_ou_sugerir_algo_sobre_o_questionario_opcional,termo_de_consentimento_livre_e_esclarecido,algum_dos_seus_familiares_de_1o_grau_filhos_irmaos_pais_ja_recebeu_diagnostico_de_transtorno_do_espectro_autista_tea,voce_tem_diagnostico_de_transtorno_de_deficit_de_atencao_e_hiperatividade_tdah,voce_considera_que_tem_alguma_condicao_ou_transtorno_mental_que_voce_se_identifica_muito_ou_tenha_sido_detectado_por_profissional_caso_a_resposta_seja_sim_descreva_tais_diagnosticos_no_campo_outro,idade,genero,escolaridade,autismo,autismo_self_perception,tdah,toc,outra_condicao,tead_total,tead_media,aq_1,aq_2,aq_3,aq_4,aq_5,aq_6,aq_7,aq_8,aq_9,aq_10,aq_total,aq_media,aq_score,soc_com,soc_inter,sensorial,rep_beh,tead_total_cfa,soc_com_m,soc_inter_m,sensorial_m,rep_beh_m,tead_total_cfa_m,asd,adhd,ocd,clinical_group,age,social_communication,social_interaction,repetitive_behavior,level_of_education
1338,07/12/2023 00:07:22,mazzali9669@gmail.com,lucca m,57 / 91,,27,masculino,uniao estavel,"pos-graduacao (especializacao, mestrado, douto...","talvez, nao sei",discordo,4.0,4.0,5.0,5.0,4.0,4.0,3.0,1.0,4.0,2.0,5.0,4.0,3.0,2.0,4.0,4.0,4.0,4.0,5.0,3.0,4.0,3.0,4.0,4.0,3.0,NaN,4.0,5.0,5.0,3.0,4.0,4.0,4.0,4.0,1.0,2.0,1.0,4.0,3.0,5.0,2.0,2.0,2.0,3.0,4.0,4.0,5.0,1.0,4.0,2.0,4.0,4.0,5.0,4.0,4.0,3.0,2.0,4.0,2.0,4.0,4.0,4.0,2.0,3.0,4.0,5.0,5.0,3.0,4.0,4.0,4.0,3.0,4.0,5.0,3.0,5.0,5.0,4.0,5.0,4.0,,declaro que li o termo de consentimento livre ...,"nao diagnosticado, porem sim",sim,tdah e tag,27.0,Masculino,Pós graduação,Não,nao. outros,Sim,Outros,Outros,NaN,NaN,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,5.0,0.5,0 - 6 points,61.0,67.0,22.0,42.0,266.0,2.772727,3.941176,4.000000,3.500000,3.546667,False,True,False,adhd,27,12.0,23.0,15.0,pos-graduacao
617,02/12/2023 12:01:45,idinei.camargo@gmail.com,idinei francisco de camargo junior,78 / 91,,42,masculino,casado(a),"pos-graduacao (especializacao, mestrado, douto...",suspeito que possa ser autista,discordo,5.0,4.0,5.0,5.0,5.0,4.0,2.0,3.0,5.0,5.0,5.0,5.0,5.0,3.0,2.0,5.0,5.0,5.0,2.0,4.0,4.0,4.0,5.0,3.0,4.0,4.0,5.0,5.0,4.0,5.0,3.0,5.0,2.0,4.0,4.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,4.0,5.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,4.0,5.0,4.0,5.0,,declaro que li o termo de consentimento livre ...,sim,nao,"fobia social, toc",42.0,Masculino,Pós graduação,Outros,sim. suspeito,Não,Outros,Outros,359.0,4.4875,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,7.0,0.7,6 or + points,90.0,77.0,24.0,60.0,335.0,4.090909,4.529412,4.500000,5.000000,4.466667,False,False,False,control,42,18.0,24.0,25.0,pos-graduacao
446,29/11/2023 20:44:57,lkunta@gmail.com,luciano,41 / 91,,44,masculino,solteiro(a),"pos-graduacao (especializacao, mestrado, douto...",nao,discordo,4.0,2.0,4.0,2.0,3.0,2.0,4.0,2.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,2.0,2.0,4.0,2.0,4.0,2.0,4.0,4.0,2.0,2.0,2.0,4.0,4.0,2.0,2.0,2.0,4.0,2.0,3.0,2.0,4.0,2.0,4.0,2.0,4.0,2.0,4.0,2.0,2.0,2.0,4.0,4.0,2.0,2.0,2.0,2.0,2.0,4.0,4.0,2.0,4.0,2.0,4.0,2.0,4.0,4.0,2.0,2.0,2.

## 3. Data analysis (DA)

Each analysis is defined once and applied to both instruments.

### 3.1 Table 1 — demographics by clinical group

In [10]:
def table1(d):
    # N, age M (SD), and categorical variables as n (%) within each clinical group
    def cat_npct(col):
        n = d.groupby("clinical_group")[col].value_counts(dropna=False)
        p = d.groupby("clinical_group")[col].value_counts(normalize=True, dropna=False).mul(100)
        return (
            pd.concat([n.rename("n"), p.rename("pct")], axis=1)
            .assign(npct=lambda x: x.apply(lambda r: f"{int(r['n'])} ({r['pct']:.1f}%)", axis=1))
            ["npct"]
            .unstack(fill_value="0 (0.0%)")
        )

    return (
        d.groupby("clinical_group")
        .agg(n=("age", "size"), age_mean=("age", "mean"), age_sd=("age", "std"))
        .round(2)
        .join(cat_npct("genero"))
        .join(cat_npct("level_of_education"))
    )

In [11]:
table1(df_tead_s)

,n,age_mean,age_sd,Feminino,Masculino,Outros,fundamental,medio,superior,pos-graduacao
clinical_group,,,,,,,,,,
adhd,69,35.06,10.67,27 (39.1%),39 (56.5%),3 (4.3%),2 (2.9%),12 (17.4%),28 (40.6%),27 (39.1%)
adhd_ocd,8,33.12,7.68,5 (62.5%),3 (37.5%),0 (0.0%),0 (0.0%),2 (25.0%),5 (62.5%),1 (12.5%)
all_conditions,6,28.67,7.99,3 (50.0%),2 (33.3%),1 (16.7%),0 (0.0%),0 (0.0%),3 (50.0%),3 (50.0%)
asd,53,34.74,10.73,20 (37.7%),30 (56.6%),3 (5.7%),1 (1.9%),3 (5.7%),22 (41.5%),27 (50.9%)
asd_adhd,37,33.19,9.98,17 (45.9%),20 (54.1%),0 (0.0%),1 (2.7%),2 (5.4%),14 (37.8%),20 (54.1%)
asd_ocd,8,35.0,9.97,4 (50.0%),4 (50.0%),0 (0.0%),0 (0.0%),0 (0.0%),5 (62.5%),3 (37.5%)
control,479,38.14,11.85,247 (51.6%),230 (48.0%),2 (0.4%),17 (3.5%),83 (17.3%),219 (45.7%),160 (33.4%)
ocd,40,41.12,12.4,17 (42.5%),22 (55.0%),1 (2.5%),2 (5.0%),6 (15.0%),17 (42.5%),15 (37.5%)


In [12]:
table1(df_team_s)

,n,age_mean,age_sd,Feminino,Outros,fundamental,medio,superior,pos-graduacao
clinical_group,,,,,,,,,
adhd,77,35.65,9.69,76 (98.7%),1 (1.3%),0 (0.0%),11 (14.3%),30 (39.0%),36 (46.8%)
adhd_ocd,14,36.43,10.7,14 (100.0%),0 (0.0%),0 (0.0%),3 (21.4%),4 (28.6%),7 (50.0%)
all_conditions,10,34.6,10.49,9 (90.0%),1 (10.0%),0 (0.0%),0 (0.0%),8 (80.0%),2 (20.0%)
asd,37,33.54,9.75,36 (97.3%),1 (2.7%),0 (0.0%),2 (5.4%),19 (51.4%),16 (43.2%)
asd_adhd,38,34.34,7.99,37 (97.4%),1 (2.6%),0 (0.0%),0 (0.0%),19 (50.0%),19 (50.0%)
asd_ocd,16,31.44,9.22,16 (100.0%),0 (0.0%),0 (0.0%),1 (6.2%),8 (50.0%),7 (43.8%)
control,467,40.21,10.61,463 (99.1%),4 (0.9%),6 (1.3%),57 (12.2%),188 (40.3%),216 (46.3%)
ocd,41,34.98,9.45,40 (97.6%),1 (2.4%),2 (4.9%),7 (17.1%),20 (48.8%),12 (29.3%)


### 3.2 Age by clinical group

OLS with `control` as the reference level.

In [13]:
def age_model(d):
    # control as reference level; rows with unparseable age are dropped by the model
    return (
        smf.ols("age ~ C(clinical_group, Treatment('control'))", data=d)
        .fit().summary2().tables[1]
        .set_axis(["coef", "se", "t", "p", "ci_low", "ci_high"], axis=1).round(3)
    )

age_model(df_tead_s)

,coef,se,t,p,ci_low,ci_high
Intercept,38.142,0.527,72.435,0.000,37.108,39.176
"C(clinical_group, Treatment('control'))[T.adhd]",-3.084,1.484,-2.078,0.038,-5.998,-0.170
"C(clinical_group, Treatment('control'))[T.adhd_ocd]",-5.017,4.108,-1.221,0.222,-13.083,3.050
"C(clinical_group, Treatment('control'))[T.all_conditions]",-9.475,4.734,-2.001,0.046,-18.771,-0.180
"C(clinical_group, Treatment('control'))[T.asd]",-3.406,1.668,-2.042,0.042,-6.682,-0.131
"C(clinical_group, Treatment('control'))[T.asd_adhd]",-4.953,1.966,-2.519,0.012,-8.814,-1.092
"C(clinical_group, Treatment('control'))[T.asd_ocd]",-3.142,4.108,-0.765,0.445,-11.208,4.925
"C(clinical_group, Treatment('control'))[T.ocd]",2.983,1.897,1.573,0.116,-0.741,6.707


In [14]:
age_model(df_team_s)

,coef,se,t,p,ci_low,ci_high
Intercept,40.208,0.474,84.746,0.000,39.276,41.139
"C(clinical_group, Treatment('control'))[T.adhd]",-4.558,1.261,-3.615,0.000,-7.034,-2.082
"C(clinical_group, Treatment('control'))[T.adhd_ocd]",-3.779,2.781,-1.359,0.175,-9.239,1.681
"C(clinical_group, Treatment('control'))[T.all_conditions]",-5.608,3.277,-1.711,0.087,-12.041,0.826
"C(clinical_group, Treatment('control'))[T.asd]",-6.667,1.751,-3.807,0.000,-10.105,-3.229
"C(clinical_group, Treatment('control'))[T.asd_adhd]",-5.866,1.730,-3.391,0.001,-9.261,-2.470
"C(clinical_group, Treatment('control'))[T.asd_ocd]",-8.770,2.607,-3.364,0.001,-13.888,-3.652
"C(clinical_group, Treatment('control'))[T.ocd]",-5.232,1.670,-3.133,0.002,-8.511,-1.953


### 3.3 Item-level ANCOVA (group + age) with model-based pairwise contrasts

Per item: `score ~ clinical_group + age`. Where the group effect is significant, pairwise contrasts come from the fitted (age-adjusted) model, Bonferroni-corrected; only significant contrasts are shown.

In [15]:
def item_ancova(d, items, alpha=0.05):
    # F-test per item first; pairwise contrasts only where the group effect holds
    fits = {it: smf.ols(f"{it} ~ C(clinical_group) + age", data=d).fit() for it in items}
    return (
        pd.concat(
            [fits[it].t_test_pairwise("C(clinical_group)", method="bonferroni").result_frame
                 .assign(item=it)
             for it in items
             if anova_lm(fits[it], typ=2).loc["C(clinical_group)", "PR(>F)"] < alpha]
        )
        .loc[lambda r: r["reject-bonferroni"]]
        .rename_axis("comparison").reset_index()
        [["item", "comparison", "coef", "Conf. Int. Low", "Conf. Int. Upp.", "pvalue-bonferroni"]]
        .sort_values(["item", "comparison"]).round(3)
    )

item_ancova(df_tead_s, items_tead)

,item,comparison,coef,Conf. Int. Low,Conf. Int. Upp.,pvalue-bonferroni
22,tead_28,control-asd,-0.544,-0.863,-0.225,0.024
23,tead_29,control-asd_adhd,-0.664,-1.061,-0.268,0.029
20,tead_3,control-asd,-0.585,-0.889,-0.280,0.005
21,tead_3,ocd-control,0.563,0.217,0.909,0.041
16,tead_33,control-asd,-0.939,-1.315,-0.563,0.000
17,tead_33,control-asd_adhd,-0.802,-1.246,-0.358,0.012
7,tead_35,asd-adhd,1.105,0.667,1.542,0.000
8,tead_35,control-asd,-1.082,-1.430,-0.734,0.000
10,tead_35,control-asd_adhd,-0.731,-1.142,-0.321,0.014
9,tead_35,ocd-asd,-0.909,-1.413,-0.405,0.012


In [16]:
item_ancova(df_team_s, items_team)

,item,comparison,coef,Conf. Int. Low,Conf. Int. Upp.,pvalue-bonferroni
9,team_1,asd_adhd-adhd,0.868,0.409,1.328,0.006
10,team_1,control-asd,-0.954,-1.354,-0.554,0.000
11,team_1,control-asd_adhd,-1.190,-1.584,-0.796,0.000
12,team_1,control-asd_ocd,-1.294,-1.888,-0.699,0.001
4,team_11,asd-adhd,0.769,0.317,1.220,0.025
5,team_11,control-asd,-1.124,-1.514,-0.735,0.000
6,team_11,control-asd_adhd,-0.927,-1.311,-0.543,0.000
7,team_11,control-asd_ocd,-1.285,-1.863,-0.706,0.000
8,team_11,ocd-control,0.662,0.292,1.032,0.013
32,team_14,control-asd,-0.868,-1.274,-0.462,0.001


### 3.4 Subscale totals — descriptives and group comparisons

M (SD) per group, plus significant pairwise contrasts (pooled-SD t-tests, Bonferroni-corrected).

In [17]:
def subscale_desc(d, subscales):
    # M (SD) of each subscale total per clinical group
    return (
        d.melt(id_vars="clinical_group", value_vars=list(subscales), var_name="subscale")
        .groupby(["subscale", "clinical_group"]).value
        .agg(lambda s: f"{s.mean():.2f} ({s.std():.2f})")
        .unstack()
    )

def subscale_pairwise(d, subscales):
    # pooled-SD pairwise contrasts, Bonferroni-corrected, significant rows only
    return (
        pd.concat(
            [smf.ols(f"{s} ~ C(clinical_group)", data=d).fit()
                 .t_test_pairwise("C(clinical_group)", method="bonferroni").result_frame
                 .assign(subscale=s)
             for s in subscales]
        )
        .loc[lambda r: r["reject-bonferroni"]]
        .rename_axis("comparison").reset_index()
        [["subscale", "comparison", "coef", "pvalue-bonferroni"]]
        .sort_values(["subscale", "comparison"]).round(3)
    )

subscale_desc(df_tead_s, subscales_tead)

clinical_group,adhd,adhd_ocd,all_conditions,asd,asd_adhd,asd_ocd,control,ocd
subscale,,,,,,,,
repetitive_behavior,18.94 (4.86),18.75 (5.18),22.83 (1.17),21.64 (2.75),21.16 (2.92),23.38 (2.20),18.12 (5.01),21.05 (3.07)
sensorial,20.51 (4.57),19.50 (4.28),22.83 (2.14),21.30 (3.46),21.86 (2.65),21.88 (3.14),17.77 (4.94),20.43 (2.84)
social_communication,15.58 (4.97),19.12 (4.05),19.83 (3.92),19.66 (3.31),19.11 (3.63),20.88 (3.48),14.67 (5.04),16.00 (4.96)
social_interaction,20.64 (4.30),20.50 (3.96),24.17 (0.98),21.81 (2.50),22.32 (2.61),23.88 (1.25),19.05 (5.04),21.25 (3.34)


In [18]:
subscale_pairwise(df_tead_s, subscales_tead)

,subscale,comparison,coef,pvalue-bonferroni
12,repetitive_behavior,asd-adhd,2.699,0.042
13,repetitive_behavior,control-asd,-3.518,0.000
14,repetitive_behavior,control-asd_adhd,-3.039,0.004
15,repetitive_behavior,control-asd_ocd,-5.252,0.044
16,repetitive_behavior,ocd-control,2.927,0.004
8,sensorial,control-adhd,-2.733,0.000
9,sensorial,control-asd,-3.527,0.000
10,sensorial,control-asd_adhd,-4.090,0.000
11,sensorial,ocd-control,2.650,0.013
0,social_communication,asd-adhd,4.081,0.000


In [19]:
subscale_desc(df_team_s, subscales_team)

clinical_group,adhd,adhd_ocd,all_conditions,asd,asd_adhd,asd_ocd,control,ocd
subscale,,,,,,,,
camouflage_ca,13.23 (4.71),14.07 (3.10),16.20 (3.39),16.57 (2.78),16.21 (3.45),17.62 (2.87),11.18 (4.49),13.78 (3.78)
camouflage_m,16.08 (4.03),17.07 (2.70),17.80 (3.49),18.16 (1.61),18.47 (1.90),19.00 (1.71),13.93 (4.76),16.29 (4.09)
gender_self_id,13.19 (4.39),13.07 (3.41),15.40 (3.81),13.11 (2.77),13.92 (3.77),13.56 (4.40),10.34 (4.18),11.68 (4.42)
sensory_sensitivity,15.58 (3.63),15.64 (3.37),17.70 (1.64),17.57 (2.33),17.74 (1.97),18.50 (1.90),13.61 (4.30),15.78 (3.51)


In [20]:
subscale_pairwise(df_team_s, subscales_team)

,subscale,comparison,coef,pvalue-bonferroni
0,camouflage_ca,asd-adhd,3.334,0.003
1,camouflage_ca,asd_adhd-adhd,2.977,0.014
2,camouflage_ca,asd_ocd-adhd,4.391,0.006
3,camouflage_ca,control-adhd,-2.056,0.003
4,camouflage_ca,control-all_conditions,-5.022,0.007
5,camouflage_ca,control-asd,-5.390,0.000
6,camouflage_ca,control-asd_adhd,-5.033,0.000
7,camouflage_ca,control-asd_ocd,-6.447,0.000
8,camouflage_ca,ocd-control,2.603,0.006
13,camouflage_m,control-adhd,-2.146,0.002


### 3.5 ASD × ADHD/OCD association

2×2 with ASD as exposure and the condition as outcome: odds ratio and risk ratio with Wald 95% CIs, plus Fisher's exact p.

In [21]:
def asd_assoc(samples, conditions=("adhd", "ocd")):
    # ASD x condition association: OR, RR, Fisher p, plus underlying 2x2 tables.
    rows = []
    tabs = {}

    for name, d in samples.items():
        for c in conditions:
            tab_df = (
                pd.crosstab(d.asd, d[c])
                .reindex(index=[True, False], columns=[True, False], fill_value=0)
            )
            tab = tab_df.to_numpy()
            t22 = Table2x2(tab)

            rows.append({
                "instrument": name,
                "condition": c.upper(),
                "odds_ratio": t22.oddsratio,
                "or_ci": tuple(round(v, 3) for v in t22.oddsratio_confint()),
                "risk_ratio": t22.riskratio,
                "rr_ci": tuple(round(v, 3) for v in t22.riskratio_confint()),
                "fisher_p": stats.fisher_exact(tab)[1],
            })

            tabs[(name, c.upper())] = (
                tab_df.rename_axis(index="asd", columns=c)
                .rename(index={True: "ASD=True", False: "ASD=False"}, columns={True: f"{c.upper()}=True", False: f"{c.upper()}=False"})
            )

    effects = pd.DataFrame(rows).round(3)
    tables = pd.concat(tabs, names=["instrument", "condition"])

    print("Association effects")
    display(effects)
    print("Underlying 2x2 tables")
    display(tables)

In [22]:
asd_assoc({"TEAD": df_tead_s, "TEAM": df_team_s})

Association effects


,instrument,condition,odds_ratio,or_ci,risk_ratio,rr_ci,fisher_p
0,TEAD,ADHD,4.751,"(3.006, 7.51)",3.200,"(2.348, 4.362)",0.00
1,TEAD,OCD,1.776,"(0.94, 3.354)",1.671,"(0.957, 2.92)",0.09
2,TEAM,ADHD,5.056,"(3.224, 7.927)",3.128,"(2.367, 4.135)",0.00
3,TEAM,OCD,3.429,"(2.028, 5.798)",2.804,"(1.849, 4.25)",0.00


Underlying 2x2 tables


ADHD=True  ADHD=False  OCD=True  OCD=False
instrument condition asd                                                  
TEAD       ADHD      ASD=True        43.0        61.0       NaN        NaN
                     ASD=False       77.0       519.0       NaN        NaN
           OCD       ASD=True         NaN         NaN      14.0       90.0
                     ASD=False        NaN         NaN      48.0      548.0
TEAM       ADHD      ASD=True        48.0        53.0       NaN        NaN
                     ASD=False       91.0       508.0       NaN        NaN
           OCD       ASD=True         NaN         NaN      26.0       75.0
                     ASD=False        NaN         NaN      55.0      544.0

In [23]:
(77+43)/(77+43+61+519)

0.17142857142857143

### 3.6 Comorbidity within ASD

Among ASD respondents: frequency of ADHD and OCD, and an exact (binomial) McNemar test on the paired flags.

In [24]:
def asd_comorbidity(samples):
    # per instrument: ASD-only subset counts and N (%) summaries + exact McNemar ADHD vs OCD
    return pd.DataFrame(
        {
            name: (
                d.loc[d.asd, ["adhd", "ocd"]]
                .pipe(
                    lambda a: pd.Series(
                        {
                            "n_asd": len(a),
                            "adhd_n_pct": f"{int(a.adhd.sum())} ({(100 * a.adhd.mean()):.1f}%)",
                            "ocd_n_pct": f"{int(a.ocd.sum())} ({(100 * a.ocd.mean()):.1f}%)",
                            "n_neither": int(
                                pd.crosstab(a.adhd, a.ocd)
                                .reindex(index=[False, True], columns=[False, True], fill_value=0)
                                .loc[False, False]
                            ),
                            "n_adhd_only": int(
                                pd.crosstab(a.adhd, a.ocd)
                                .reindex(index=[False, True], columns=[False, True], fill_value=0)
                                .loc[True, False]
                            ),
                            "n_ocd_only": int(
                                pd.crosstab(a.adhd, a.ocd)
                                .reindex(index=[False, True], columns=[False, True], fill_value=0)
                                .loc[False, True]
                            ),
                            "n_both": int(
                                pd.crosstab(a.adhd, a.ocd)
                                .reindex(index=[False, True], columns=[False, True], fill_value=0)
                                .loc[True, True]
                            ),
                            "mcnemar_p": round(
                                mcnemar(
                                    pd.crosstab(a.adhd, a.ocd).reindex(
                                        index=[False, True], columns=[False, True], fill_value=0
                                    )
                                ).pvalue,
                                3,
                            ),
                        }
                    )
                )
            )
            for name, d in samples.items()
        }
    )


asd_comorbidity({"TEAD": df_tead_s, "TEAM": df_team_s})

,TEAD,TEAM
n_asd,104,101
adhd_n_pct,43 (41.3%),48 (47.5%)
ocd_n_pct,14 (13.5%),26 (25.7%)
n_neither,53,37
n_adhd_only,37,38
n_ocd_only,8,16
n_both,6,10
mcnemar_p,0.0,0.004


In [25]:
# R-like effect summary table (odds ratio + matched-pairs OR)
def asd_comorbidity_effects(samples, var_a, var_b, continuity=0.5):

    out = []

    for name, d in samples.items():
        if var_a not in d.columns or var_b not in d.columns:
            missing = [v for v in (var_a, var_b) if v not in d.columns]
            raise KeyError(f"{name}: missing column(s): {missing}")

        # Restrict to ASD participants; inference is conditional on ASD.
        a = d.loc[d.asd, [var_a, var_b]].dropna()
        
        # Display ASD-only paired 2x2 table: True first (epidemiological convention).
        tab_with_totals = (
            pd.crosstab(a[var_a], a[var_b], margins=True, margins_name="Total")
            .reindex(index=[True, False, "Total"], columns=[True, False, "Total"], fill_value=0)
        )
        print(f"{name} (ASD-only) 2x2 table: {var_a} (rows) x {var_b} (columns)")
        display(tab_with_totals)

        # Paired 2x2 table for var_a vs var_b within the same participants.
        tab = tab_with_totals.reindex(index=[False, True], columns=[False, True], fill_value=0)

        n_asd = len(a)
        n_a = int(a[var_a].sum())
        n_b = int(a[var_b].sum())

        # Conditional risks: P(var_a|ASD) and P(var_b|ASD).
        risk_a = n_a / n_asd
        risk_b = n_b / n_asd
        risk_ratio = risk_a / risk_b if risk_b > 0 else np.nan

        # Discordant cells used by McNemar and matched-pairs OR:
        # b = var_a only, c = var_b only.
        b = int(tab.loc[True, False])
        c = int(tab.loc[False, True])
        matched_or = (b / c) if c > 0 else np.inf

        # Continuity correction produces finite OR/CI for small samples or zero cells.
        b_cc, c_cc = b + continuity, c + continuity
        matched_or_cc = b_cc / c_cc
        log_or = np.log(matched_or_cc)
        se_log_or = np.sqrt((1 / b_cc) + (1 / c_cc))
        ci_low = np.exp(log_or - 1.96 * se_log_or)
        ci_high = np.exp(log_or + 1.96 * se_log_or)

        out.append({
            "sample": name,
            "comparison": f"{var_a} vs {var_b}",
            "n_asd": n_asd,
            f"{var_a}_n_pct": f"{n_a} ({100 * risk_a:.1f}%)",
            f"{var_b}_n_pct": f"{n_b} ({100 * risk_b:.1f}%)",
            f"risk_ratio_{var_a}_vs_{var_b}": risk_ratio,
            f"n_{var_a}_only": b,
            f"n_{var_b}_only": c,
            "matched_or_b_over_c": matched_or,
            "matched_or_cc": matched_or_cc,
            "matched_or_ci95_low": ci_low,
            "matched_or_ci95_high": ci_high,
            "mcnemar_exact_p": mcnemar(tab, exact=True).pvalue,
        })

    return pd.DataFrame(out)

In [26]:
asd_comorbidity_effects(
    {"TEAD": df_tead_s, "TEAM": df_team_s},
    var_a="adhd",
    var_b="ocd",
)

TEAD (ASD-only) 2x2 table: adhd (rows) x ocd (columns)


ocd,True,False,Total
adhd,,,
True,6,37,43
False,8,53,61
Total,14,90,104


TEAM (ASD-only) 2x2 table: adhd (rows) x ocd (columns)

ocd,True,False,Total
adhd,,,
True,10,38,48
False,16,37,53
Total,26,75,101


,sample,comparison,n_asd,adhd_n_pct,ocd_n_pct,risk_ratio_adhd_vs_ocd,n_adhd_only,n_ocd_only,matched_or_b_over_c,matched_or_cc,matched_or_ci95_low,matched_or_ci95_high,mcnemar_exact_p
0,TEAD,adhd vs ocd,104,43 (41.3%),14 (13.5%),3.071429,37,8,4.625,4.411765,2.095301,9.289198,0.000015
1,TEAM,adhd vs ocd,101,48 (47.5%),26 (25.7%),1.846154,38,16,2.375,2.333333,1.310722,4.153777,0.003838


## 4. Specificity — which behaviors separate ASD from the other conditions

Core analysis for the manuscript aim. Per item: Cohen's *d* and AUC of pure ASD vs. each other pure group (control, ADHD, OCD). An autism-specific item shows large *d*/AUC against ADHD and OCD, not only against controls. Sorted by *d* vs. ADHD, the hardest contrast.

In [27]:
def specificity(d, items, ref="asd", comparisons=("control", "adhd", "ocd")):
    # Cohen's d (pooled SD) and AUC per item: pure ASD vs each pure comparison group
    def one(it, grp):
        # clinical_group labels are mutually exclusive, so "asd" and "adhd" are ASD-only/ADHD-only
        x, y = (d.loc[d.clinical_group == g, it].dropna() for g in (ref, grp))
        return {
            "item": it,
            "vs": grp,
            "cohens_d": (x.mean() - y.mean())
                / np.sqrt(((len(x)-1)*x.var() + (len(y)-1)*y.var()) / (len(x)+len(y)-2)),
            "auc": stats.mannwhitneyu(x, y).statistic / (len(x) * len(y)),
        }

    # M (SD), N for each item within each pure group (ref + all comparison groups)
    msd_groups = (ref, *comparisons)
    msd = (
        d.melt(id_vars="clinical_group", value_vars=items, var_name="item", value_name="score")
        .loc[lambda x: x["clinical_group"].isin(msd_groups)]
        .groupby(["item", "clinical_group"])["score"]
        .agg(lambda s: f"{s.mean():.2f} ({s.std():.2f}), N={int(s.count())}")
        .unstack()
        .reindex(columns=msd_groups)
    )
    msd.columns = pd.MultiIndex.from_product([["m_sd"], msd.columns])

    effects = (
        pd.DataFrame([one(it, g) for it in items for g in comparisons])
        .pivot(index="item", columns="vs", values=["cohens_d", "auc"])
    )

    return (
        msd.join(effects)
        .sort_values(("cohens_d", "adhd"), ascending=False)
        .round(2)
    )

specificity(df_tead_s, items_tead)

m_sd                                         \
                       asd             control               adhd   
item                                                                
tead_8   3.66 (1.00), N=53  2.54 (1.17), N=479  2.48 (1.16), N=69   
tead_35  3.77 (0.97), N=53  2.67 (1.25), N=479  2.67 (1.29), N=69   
tead_52  4.43 (0.67), N=53  3.70 (1.22), N=479  3.70 (1.19), N=69   
tead_36  4.25 (0.68), N=53  3.21 (1.20), N=479  3.57 (1.18), N=69   
tead_56  4.34 (0.71), N=53  3.47 (1.29), N=479  3.70 (1.18), N=69   
tead_60  4.66 (0.59), N=53  4.00 (1.12), N=479  4.07 (1.13), N=69   
tead_33  3.58 (1.23), N=53  2.62 (1.33), N=479  2.93 (1.38), N=69   
tead_38  4.40 (0.66), N=53  3.62 (1.28), N=479  3.94 (1.34), N=69   
tead_70  4.55 (0.75), N=53  3.78 (1.25), N=479  4.23 (0.96), N=69   
tead_59  4.08 (0.90), N=53  3.52 (1.26), N=479  3.67 (1.30), N=69   
tead_29  4.26 (0.84), N=53  3.65 (1.29), N=479  3.88 (1.30), N=69   
tead_3   4.64 (0.52), N=53  4.01 (1.20), N=479  4.41 (0.91), N=69   
tead_9   4.60 (0.53), N=53  4.03 (1.16), N=479  4.39 (0.91), N=69   
tead_63  4.13 (1.02), N=53  3.43 (1.29), N=479  3.81 (1.29), N=69   
tead_28  4.43 (0.72), N=53  3.88 (1.20), N=479  4.19 (1.14), N=69   
tead_4   4.00 (1.16), N=53  3.02 (1.40), N=479  3.74 (1.41), N=69   
tead_69  4.26 (0.98), N=53  3.63 (1.24), N=479  4.12 (1.19), N=69   
tead_5   3.87 (1.13), N=53  3.49 (1.28), N=479  3.77 (1.18), N=69   
tead_73  4.23 (0.87), N=53  3.56 (1.33), N=479  4.19 (1.03), N=69   
tead_66  4.26 (0.88), N=53  3.79 (1.21), N=479  4.23 (1.13), N=69   

                           cohens_d                 auc                
                       ocd     adhd control   ocd  adhd control   ocd  
item                                                                   
tead_8   2.75 (1.21), N=40     1.08    0.97  0.83  0.77    0.76  0.71  
tead_35  2.83 (1.24), N=40     0.95    0.90  0.87  0.74    0.74  0.72  
tead_52  4.53 (0.55), N=40     0.74    0.63 -0.15  0.68    0.67  0.47  
tead_36  3.75 (1.06), N=40     0.68    0.89  0.58  0.66    0.75  0.62  
tead_56  4.33 (0.86), N=40     0.64    0.70  0.02  0.65    0.69  0.49  
tead_60  4.53 (0.78), N=40     0.63    0.61  0.20  0.65    0.67  0.53  
tead_33  2.83 (1.38), N=40     0.50    0.73  0.59  0.64    0.70  0.66  
tead_38  3.85 (1.14), N=40     0.41    0.63  0.61  0.55    0.67  0.62  
tead_70  4.28 (0.78), N=40     0.36    0.63  0.36  0.59    0.68  0.61  
tead_59  4.05 (1.08), N=40     0.36    0.45  0.03  0.57    0.61  0.48  
tead_29  4.05 (0.93), N=40     0.34    0.49  0.24  0.55    0.63  0.57  
tead_3   4.53 (0.93), N=40     0.31    0.55  0.16  0.54    0.64  0.49  
tead_9   4.47 (0.85), N=40     0.28    0.52  0.19  0.54    0.63  0.52  
tead_63  3.62 (1.19), N=40     0.27    0.55  0.46  0.56    0.66  0.62  
tead_28  4.25 (0.87), N=40     0.25    0.48  0.23  0.52    0.62  0.55  
tead_4   3.75 (1.17), N=40     0.20    0.71  0.21  0.54    0.70  0.57  
tead_69  4.03 (0.92), N=40     0.13    0.52  0.25  0.52    0.65  0.59  
tead_5   3.95 (1.08), N=40     0.09    0.30 -0.07  0.52    0.58  0.48  
tead_73  4.10 (1.03), N=40     0.04    0.52  0.13  0.49    0.64  0.52  
tead_66  4.28 (1.01), N=40     0.03    0.40 -0.01  0.48    0.61  0.48

In [28]:
specificity(df_team_s, items_team)

m_sd                                         \
                       asd             control               adhd   
item                                                                
team_9   3.81 (0.94), N=37  2.44 (1.22), N=467  2.86 (1.35), N=77   
team_11  4.46 (0.80), N=37  3.12 (1.28), N=467  3.62 (1.24), N=77   
team_6   4.27 (0.90), N=37  2.77 (1.33), N=467  3.43 (1.39), N=77   
team_35  4.76 (0.49), N=37  3.71 (1.31), N=467  4.19 (1.05), N=77   
team_21  4.59 (0.55), N=37  3.52 (1.31), N=467  4.04 (1.13), N=77   
team_32  4.62 (0.79), N=37  3.87 (1.24), N=467  4.06 (1.07), N=77   
team_1   4.03 (0.96), N=37  2.85 (1.27), N=467  3.32 (1.38), N=77   
team_22  4.51 (0.51), N=37  3.41 (1.37), N=467  3.96 (1.20), N=77   
team_14  4.51 (0.51), N=37  3.48 (1.33), N=467  3.99 (1.22), N=77   
team_7   4.54 (0.56), N=37  3.52 (1.34), N=467  4.09 (1.03), N=77   
team_37  4.14 (1.18), N=37  3.04 (1.46), N=467  3.64 (1.34), N=77   
team_5   4.46 (0.69), N=37  3.46 (1.32), N=467  4.09 (1.10), N=77   
team_36  4.05 (1.03), N=37  2.98 (1.45), N=467  3.69 (1.17), N=77   
team_25  2.43 (1.30), N=37  1.81 (1.17), N=467  2.26 (1.43), N=77   
team_3   3.19 (1.22), N=37  2.54 (1.42), N=467  3.49 (1.39), N=77   
team_4   3.03 (1.30), N=37  2.53 (1.40), N=467  3.35 (1.40), N=77   

                           cohens_d                 auc                
                       ocd     adhd control   ocd  adhd control   ocd  
item                                                                   
team_9   2.76 (1.22), N=41     0.77    1.14  0.96  0.70    0.80  0.74  
team_11  3.95 (1.07), N=41     0.75    1.07  0.53  0.71    0.81  0.64  
team_6   3.54 (1.34), N=41     0.67    1.15  0.63  0.67    0.81  0.65  
team_35  4.44 (0.84), N=41     0.62    0.83  0.46  0.66    0.75  0.60  
team_21  4.20 (1.10), N=41     0.57    0.84  0.45  0.63    0.74  0.58  
team_32  4.37 (1.02), N=41     0.56    0.61  0.28  0.66    0.69  0.56  
team_1   3.54 (1.10), N=41     0.56    0.94  0.47  0.64    0.76  0.63  
team_22  3.95 (1.26), N=41     0.54    0.83  0.57  0.60    0.73  0.59  
team_14  4.02 (1.21), N=41     0.50    0.80  0.52  0.59    0.73  0.58  
team_7   4.12 (1.05), N=41     0.50    0.79  0.49  0.61    0.73  0.60  
team_37  3.34 (1.44), N=41     0.39    0.76  0.60  0.61    0.72  0.66  
team_5   3.95 (1.18), N=41     0.37    0.78  0.52  0.58    0.72  0.61  
team_36  3.63 (1.30), N=41     0.32    0.75  0.36  0.59    0.71  0.59  
team_25  1.98 (1.27), N=41     0.12    0.53  0.35  0.55    0.64  0.61  
team_3   2.93 (1.49), N=41    -0.23    0.46  0.19  0.43    0.64  0.56  
team_4   2.83 (1.39), N=41    -0.24    0.36  0.15  0.43    0.61  0.54

### 4.1 Class-specificity map (ASD, ADHD, OCD) using pairwise d + one-vs-rest AUC


In [29]:
def cohens_d(x, y):
    x = pd.Series(x).dropna()
    y = pd.Series(y).dropna()
    if len(x) < 2 or len(y) < 2:
        return np.nan
    v = ((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2)
    if v <= 0 or np.isnan(v):
        return np.nan
    return (x.mean() - y.mean()) / np.sqrt(v)


def one_vs_rest_auc(d, item, target):
    s = d[["clinical_group", item]].dropna()
    x = s[item]
    y = (s["clinical_group"] == target).astype(int)
    n_pos = int(y.sum())
    n_neg = int((1 - y).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    u = stats.mannwhitneyu(x[y == 1], x[y == 0], alternative="two-sided").statistic
    return u / (n_pos * n_neg)


def classify_symbol(min_d, auc):
    # Rules: ✓ strong class-specific signal; ~ partial/overlap; X weak or opposite
    if pd.notna(min_d) and pd.notna(auc) and (min_d >= 0.30) and (auc >= 0.60):
        return "✓"
    if pd.notna(min_d) and pd.notna(auc) and (min_d > -0.20) and (auc >= 0.55):
        return "~"
    return "X"


def class_specificity_map(d, items, classes=("asd", "adhd", "ocd"), item_labels=None):
    rows = []
    for it in items:
        row = {"item": item_labels.get(it, it) if item_labels else it}
        for c in classes:
            others = [o for o in classes if o != c]
            ds = []
            for o in others:
                x = d.loc[d.clinical_group == c, it]
                y = d.loc[d.clinical_group == o, it]
                ds.append(cohens_d(x, y))
            min_d = np.nanmin(ds) if np.any(pd.notna(ds)) else np.nan
            auc = one_vs_rest_auc(d, it, c)
            row[c.upper()] = classify_symbol(min_d, auc)
            row[f"{c}_min_d"] = min_d
            row[f"{c}_ovr_auc"] = auc
        rows.append(row)

    out = pd.DataFrame(rows)
    symbols = out[["item", "ASD", "ADHD", "OCD"]]
    diagnostics = out[[
        "item",
        "asd_min_d", "asd_ovr_auc",
        "adhd_min_d", "adhd_ovr_auc",
        "ocd_min_d", "ocd_ovr_auc",
    ]].round(3)
    return symbols, diagnostics


symbols_tead, diagnostics_tead = class_specificity_map(
    df_tead_s,
    items_tead,
    classes=("asd", "adhd", "ocd"),
    item_labels=item_labels_tead,
)

symbols_team, diagnostics_team = class_specificity_map(
    df_team_s,
    items_team,
    classes=("asd", "adhd", "ocd"),
    item_labels=item_labels_team,
)

print("TEAD class-specificity map")
display(symbols_tead)
display(diagnostics_tead)

print("TEAM class-specificity map")
display(symbols_team)
display(diagnostics_team)

TEAD class-specificity map


,item,ASD,ADHD,OCD
0,8 - I have trouble reading facial expressions,✓,X,X
1,35 - I find it hard to read other people's bod...,✓,X,X
2,36 - I often miss social cues.,✓,X,X
3,38 - I find it hard to know when someone wants...,✓,X,X
4,33 - I have difficulty understanding non-liter...,✓,X,X
5,"9 - Compared to other people, I often feel out...",~,X,~
6,3 - I often become exhausted from social inter...,~,X,~
7,5 - I often try to avoid interacting with peop...,~,X,~
8,28 - I enjoy social interaction less than othe...,~,X,X
9,29 - Others have told me I seem withdrawn or d...,~,X,X


,item,asd_min_d,asd_ovr_auc,adhd_min_d,adhd_ovr_auc,ocd_min_d,ocd_ovr_auc
0,8 - I have trouble reading facial expressions,0.830,0.733,-1.082,0.439,-0.830,0.508
1,35 - I find it hard to read other people's bod...,0.866,0.721,-0.951,0.457,-0.866,0.499
2,36 - I often miss social cues.,0.576,0.713,-0.684,0.538,-0.576,0.576
3,38 - I find it hard to know when someone wants...,0.414,0.630,-0.414,0.557,-0.607,0.506
4,33 - I have difficulty understanding non-liter...,0.500,0.669,-0.500,0.523,-0.587,0.500
5,"9 - Compared to other people, I often feel out...",0.188,0.593,-0.276,0.548,-0.188,0.570
6,3 - I often become exhausted from social inter...,0.160,0.606,-0.307,0.559,-0.160,0.604
7,5 - I often try to avoid interacting with peop...,-0.074,0.554,-0.159,0.530,0.074,0.573
8,28 - I enjoy social interaction less than othe...,0.233,0.597,-0.250,0.563,-0.233,0.543
9,29 - Others have told me I seem withdrawn or d...,0.244,0.599,-0.338,0.533,-0.244,0.537


TEAM class-specificity map


,item,ASD,ADHD,OCD
0,6 - I imitate other people when I am interacti...,✓,X,X
1,11 - I imitate other people's hand gestures wh...,✓,X,X
2,1 - I imitate other people's behavior when I a...,✓,X,X
3,9 - I imitate how people talk or speak when I ...,✓,X,X
4,"5 - As a child (ages 5-12), other children tho...",✓,X,X
5,3 - I found it very difficult to make friends ...,X,~,X
6,"4 - As a child (ages 5-12), I was not interest...",X,~,X
7,"25 - As a child (ages 5-12), I felt unsure abo...",~,~,X
8,"14 - I often feel that I am ""playing a role"" i...",✓,X,X
9,"22 - I often ""pretend"" to be like other people.",✓,X,X


,item,asd_min_d,asd_ovr_auc,adhd_min_d,adhd_ovr_auc,ocd_min_d,ocd_ovr_auc
0,6 - I imitate other people when I am interacti...,0.635,0.750,-0.671,0.571,-0.635,0.591
1,11 - I imitate other people's hand gestures wh...,0.533,0.758,-0.750,0.548,-0.533,0.623
2,1 - I imitate other people's behavior when I a...,0.475,0.700,-0.557,0.546,-0.475,0.586
3,9 - I imitate how people talk or speak when I ...,0.772,0.753,-0.772,0.536,-0.963,0.515
4,"5 - As a child (ages 5-12), other children tho...",0.373,0.672,-0.373,0.594,-0.518,0.556
5,3 - I found it very difficult to make friends ...,-0.227,0.585,0.227,0.650,-0.397,0.525
6,"4 - As a child (ages 5-12), I was not interest...",-0.236,0.568,0.236,0.639,-0.372,0.526
7,"25 - As a child (ages 5-12), I felt unsure abo...",0.124,0.607,-0.124,0.557,-0.355,0.494
8,"14 - I often feel that I am ""playing a role"" i...",0.504,0.668,-0.504,0.565,-0.516,0.570
9,"22 - I often ""pretend"" to be like other people.",0.538,0.680,-0.538,0.570,-0.573,0.571
